## linear model in latex

In [20]:
import numpy as np
import pandas as pd
import spreg
import geopandas as gpd
from statsmodels.stats.outliers_influence import summary_table
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [17]:
gdf = gpd.read_file("G:/My Drive/INVESTIGACION/POSDOC/Data/Vector/df_catchments_kmeans.gpkg")
#gdf.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 526 entries, 0 to 525
Data columns (total 22 columns):
 #   Column               Non-Null Count  Dtype   
---  ------               --------------  -----   
 0   id                   526 non-null    int64   
 1   Nombre               526 non-null    object  
 2   ID_CUENCA            526 non-null    float64 
 3   cuenca               526 non-null    object  
 4   area                 526 non-null    int64   
 5   elev_mean            526 non-null    float64 
 6   elev_median          526 non-null    float64 
 7   rel_mean             526 non-null    float64 
 8   rel_median           526 non-null    float64 
 9   rainfallAnnual_mean  526 non-null    float64 
 10  Densidad             526 non-null    float64 
 11  hypso_inte           526 non-null    float64 
 12  slope_mean           526 non-null    float64 
 13  kmeans               526 non-null    object  
 14  RainfallDaysmean     526 non-null    float64 
 15  RainfallDaysmed

In [55]:
# Apply log transformation
gdf["y_log"] = np.log(gdf['lands_rec'] + 1)

# Define dependent and independent variables
landslides = gdf["y_log"]
y = np.array(landslides).reshape(-1, 1)

# Select and standardize independent variables
var = ['area', 'hypso_inte', 'Densidad', 'rainfallAnnual_mean', 'elev_mean', 'slope_mean', 'rel_mean']
st = StandardScaler()
dfs = pd.DataFrame(st.fit_transform(gdf[var]), index=gdf.index, columns=var)

# Add the transformed and other necessary columns to `dfs`
dfs["y_log"] = gdf["y_log"]
dfs["knn5"] = gdf["knn5"]
dfs["basin"] = gdf["cuenca"]

# One-hot encode categorical variables
encoder = OneHotEncoder(drop='first', dtype=int)
encoded_columns = encoder.fit_transform(gdf[['landcovermedian', 'geomedian']]).toarray()
encoded_columns_df = pd.DataFrame(encoded_columns, index=gdf.index, columns=encoder.get_feature_names_out(['landcovermedian', 'geomedian']))

# Merge the encoded columns with the standardized DataFrame
dfs = pd.concat([dfs, encoded_columns_df], axis=1)


In [63]:
# Define variable combinations for models
model_vars = [
    ['area', 'hypso_inte', 'Densidad', 'rainfallAnnual_mean', 'elev_mean', 'slope_mean', 'rel_mean'] + list(encoded_columns_df.columns),
    ['hypso_inte', 'Densidad', 'rainfallAnnual_mean', 'slope_mean'],
    ['hypso_inte', 'Densidad', 'rainfallAnnual_mean', 'rel_mean'] + list(encoded_columns_df.columns),
    ["area", "slope_mean", "rel_mean"] + list(encoded_columns_df.columns),
    ["area", 'elev_mean', "rel_mean"]
]

# Map for LaTeX-friendly variable names
var_map = {
    'const': 'Constant',
    'area': '$A$',
    'hypso_inte': '$HI$',
    'Densidad': '$L_d$',
    'rainfallAnnual_mean': '$P$',
    'elev_mean': '$E$',
    'slope_mean': '$S$',
    'rel_mean': '$H$',
    'landcovermedian_grass': '$L_c(grass)$',
    'geomedian_sediment': '$G$ (sediment)',
    'geomedian_volcanic': '$G$ (volcanic)'
}

# Map for model names
model_map = {
    'Modelo 1': 'All Covs.',
    'Modelo 2': 'Covariates',
    'Modelo 3': 'Covariates',
    'Modelo 4': 'Covariates',
    'Modelo 5': 'Selected Covs.'
}

In [65]:
# Initialize results DataFrame
results_df = pd.DataFrame(columns=["Variable"] + [f"Modelo {i+1}" for i in range(len(model_vars))])
results_df["Variable"] = ['const'] + var + list(encoded_columns_df.columns) + ["Adj R2", "AIC"]

# Iterate through the models and fit OLS
for i, vars_subset in enumerate(model_vars):
    # Add constant to independent variables
    X_with_const = np.hstack((np.ones((y.shape[0], 1)), pd.get_dummies(dfs[vars_subset], drop_first=True).astype(float).to_numpy()))
    
    # Fit the OLS model
    model = spreg.OLS(y, X_with_const, name_y="landslides", name_x=['const'] + vars_subset)
    
    # Save coefficients and statistics
    used_vars = ['const'] + vars_subset  # Variables used in this model
    for variable in ['const'] + var + list(encoded_columns_df.columns):
        if variable in used_vars:
            idx = used_vars.index(variable)
            coef = model.betas[idx][0]
            std_err = model.std_err[idx]
            p_value = model.t_stat[idx][1]
            # Format values for LaTeX: bold the significant values
            coef_str = f"{coef:.3f} ({std_err:.3f})"
            if p_value < 0.05:
                coef_str = f"\\textbf{{{coef:.3f}}} ({std_err:.3f})"  # Bold in LaTeX
            results_df.loc[results_df["Variable"] == variable, f"Modelo {i+1}"] = coef_str
        else:
            # Replace with '-' when the variable is not used in the model
            results_df.loc[results_df["Variable"] == variable, f"Modelo {i+1}"] = '-'
    
    # Save Adjusted R2 and AIC
    results_df.loc[results_df["Variable"] == "Adj R2", f"Modelo {i+1}"] = f"{model.ar2:.3f}"
    results_df.loc[results_df["Variable"] == "AIC", f"Modelo {i+1}"] = f"{model.aic:.3f}"

# Rename the columns for the models
results_df.columns = ["Variable"] + [model_map.get(f"Modelo {i+1}", f"Modelo {i+1}") for i in range(len(model_vars))]

# Apply var_map to the Variable column
results_df["Variable"] = results_df["Variable"].map(var_map).fillna(results_df["Variable"])

# Generate LaTeX table with bold column headers and custom formatting
latex_table = results_df.to_latex(index=False, escape=False, header=True)

# Insert a midrule before "Adj. $R^2$"
latex_table = latex_table.replace("Adj R2", "\\midrule\nAdj. $R^2$")

# Add the note for p-values after the bottomrule
latex_table = latex_table.replace("\\bottomrule", "\\bottomrule\n\\multicolumn{6}{l}{\\textbf{$p<0.05$}, (standard errors)}")

# Bold the column headers
latex_table = latex_table.replace("\\toprule\nVariable", "\\toprule\n\\textbf{Variable}")

# Replace model names with bolded ones in the header
for model_name in results_df.columns[1:]:
    latex_table = latex_table.replace(model_name, f"\\textbf{{{model_name}}}")

print(latex_table)



\begin{tabular}{llllll}
\toprule
\textbf{Variable} & \textbf{All Covs.} & \textbf{\textbf{\textbf{Covariates}}} & \textbf{\textbf{\textbf{Covariates}}} & \textbf{\textbf{\textbf{Covariates}}} & \textbf{Selected Covs.} \\
\midrule
Constant & \textbf{1.814} (0.074) & \textbf{1.840} (0.056) & \textbf{1.862} (0.086) & \textbf{1.759} (0.076) & \textbf{1.840} (0.048) \\
$A$ & \textbf{0.586} (0.050) & - & - & \textbf{0.603} (0.051) & \textbf{0.557} (0.049) \\
$HI$ & 0.069 (0.058) & 0.104 (0.063) & 0.035 (0.067) & - & - \\
$L_d$ & 0.179 (0.150) & \textbf{-0.449} (0.142) & -0.024 (0.106) & - & - \\
$P$ & -0.077 (0.067) & \textbf{-0.253} (0.063) & \textbf{-0.272} (0.066) & - & - \\
$E$ & \textbf{0.423} (0.074) & - & - & - & \textbf{0.514} (0.058) \\
$S$ & -0.123 (0.298) & \textbf{1.112} (0.146) & - & \textbf{0.415} (0.191) & - \\
$H$ & \textbf{0.573} (0.204) & - & \textbf{0.792} (0.103) & \textbf{0.492} (0.182) & \textbf{0.546} (0.058) \\
$L_c(grass)$ & 0.202 (0.123) & - & 0.197 (0.143) & \textb